In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import re


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressi ng Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/african-folktales-slm-challenge/sample_submission.csv
/kaggle/input/competitions/african-folktales-slm-challenge/train_prompts.csv
/kaggle/input/competitions/african-folktales-slm-challenge/documents.csv
/kaggle/input/competitions/african-folktales-slm-challenge/dataset-metadata.json
/kaggle/input/competitions/african-folktales-slm-challenge/test_prompts.csv
/kaggle/input/competitions/african-folktales-slm-challenge/baseline_submission.csv


In [2]:
DATA_DIR = "/kaggle/input/competitions/african-folktales-slm-challenge"

documents = pd.read_csv(
    os.path.join(DATA_DIR, "documents.csv")
)

train = pd.read_csv(
    os.path.join(DATA_DIR, "train_prompts.csv")
)

test = pd.read_csv(
    os.path.join(DATA_DIR, "test_prompts.csv")
)

sample_submission = pd.read_csv(
    os.path.join(DATA_DIR, "sample_submission.csv")
)

baseline = pd.read_csv(
    os.path.join(DATA_DIR, "baseline_submission.csv")
)

In [3]:
print("Documents:", documents.shape)
print("Train:", train.shape)
print("Test:", test.shape)

display(documents.head())
display(train.head())
display(test.head())

Documents: (24, 8)
Train: (38, 6)
Test: (10, 4)


,document_id,title,theme,culture_region,text,origin,source_url,license
0,doc_tri_001,The spider and the shared pot,trickster,west_africa,"When famine visited the village, the spider tr...",synthetic,NaN,CC0-1.0
1,doc_tri_002,Tortoise and the river festival,trickster,southern_africa,Tortoise could not swim but wished to attend t...,synthetic,NaN,CC0-1.0
2,doc_tri_003,Hare and the drum of thunder,trickster,east_africa,Hare found a hollow log that echoed like thund...,synthetic,NaN,CC0-1.0
3,doc_ori_001,Why the baobab looks upside down,origin_myth,east_africa,The first baobab boasted that its roots could ...,synthetic,NaN,CC0-1.0
4,doc_ori_002,How the river learned to bend,origin_myth,central_africa,Long ago the river ran straight and forgot the...,synthetic,NaN,CC0-1.0


,prompt,theme,culture_region,document_id,reference_story,PromptId
0,Explain in story form why the baobab looks ups...,origin_myth,east_africa,doc_ori_001,The first baobab boasted that its roots could ...,1
1,Tell a story showing generosity warms a commun...,moral_tale,east_africa,doc_mor_003,A traveler asked for one ladle of stew and was...,2
2,Tell an origin myth about a river learning to ...,origin_myth,central_africa,doc_ori_002,Long ago the river ran straight and forgot the...,3
3,Medicine bark across three valleys — hero tale.,hero_journey,central_africa,doc_her_001,She braided grass ropes with strangers in a st...,4
4,Write a fable where elephant learns from firef...,animal_fable,central_africa,doc_ani_002,Elephant demanded the forest paths be cleared ...,5


,PromptId,prompt,theme,culture_region
0,1001,Tell a moral market tale about returning a los...,moral_tale,west_africa
1,1002,Create a hero story of an orphan who saves fis...,hero_journey,east_africa
2,1003,Create an East African tale of hare claiming t...,trickster,east_africa
3,1004,Hyena jumps at the moon in water.,animal_fable,east_africa
4,1005,Tell a trickster tale of monkey guarding a hon...,trickster,central_africa


In [4]:
print("Themes:")
print(documents["theme"].value_counts())

print("\nRegions:")
print(documents["culture_region"].value_counts())

Themes:
theme
trickster           4
origin_myth         4
moral_tale          4
hero_journey        4
animal_fable        4
community_wisdom    4
Name: count, dtype: int64

Regions:
culture_region
west_africa        6
east_africa        6
southern_africa    5
central_africa     5
diaspora           2
Name: count, dtype: int64


In [5]:
print(test[[
    "PromptId",
    "prompt",
    "theme",
    "culture_region"
]])

   PromptId                                             prompt  \
0      1001  Tell a moral market tale about returning a los...   
1      1002  Create a hero story of an orphan who saves fis...   
2      1003  Create an East African tale of hare claiming t...   
3      1004                  Hyena jumps at the moon in water.   
4      1005  Tell a trickster tale of monkey guarding a hon...   
5      1006  Write an origin myth about stars spilling from...   
6      1007      Grandmother sings and the river bends — myth.   
7      1008         Brothers, one path, drought — what lesson?   
8      1009  Tell a hero tale of a drummer who brings rain ...   
9      1010               Harvest dance drum picks the leader.   

              theme   culture_region  
0        moral_tale      west_africa  
1      hero_journey      east_africa  
2         trickster      east_africa  
3      animal_fable      east_africa  
4         trickster   central_africa  
5       origin_myth  southern_africa  


In [6]:
def clean_text(text):
    text = str(text).lower()

    # Remove punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
documents["retrieval_text"] = (
    documents["title"].fillna("") + " " +
    documents["text"].fillna("") + " " +
    documents["theme"].fillna("") + " " +
    documents["culture_region"].fillna("")
)

documents["retrieval_text"] = documents["retrieval_text"].apply(clean_text)


In [8]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

In [9]:
document_vectors = vectorizer.fit_transform(
    documents["retrieval_text"]
)

print(document_vectors.shape)


(24, 1827)


In [10]:
test["retrieval_text"] = (
    test["prompt"].fillna("") + " " +
    test["theme"].fillna("") + " " +
    test["culture_region"].fillna("")
)

test["retrieval_text"] = test["retrieval_text"].apply(clean_text)

In [11]:
test_vectors = vectorizer.transform(
    test["retrieval_text"]
)

similarities = cosine_similarity(
    test_vectors,
    document_vectors
)
print (similarities[0])

[0.02453536 0.02398517 0.00258948 0.0027518  0.00302114 0.02944446
 0.33868343 0.07026865 0.07415487 0.00279721 0.02763248 0.00258673
 0.00268887 0.00296438 0.00288934 0.00289439 0.02959781 0.05913355
 0.00272519 0.00283235 0.07316022 0.         0.02925877 0.00314073]


In [12]:
results = []  

for i, row in test.iterrows():

    scores = similarities[i]

    best_index = np.argmax(scores)

    best_document = documents.iloc[best_index]

    results.append({
        "PromptId": row["PromptId"],
        "prompt": row["prompt"],
        "theme": row["theme"],
        "culture_region": row["culture_region"],
        "document_id": best_document["document_id"],
        "document_title": best_document["title"],
        "similarity": scores[best_index]
    })

matches = pd.DataFrame(results)

display(matches)

,PromptId,prompt,theme,culture_region,document_id,document_title,similarity
0,1001,Tell a moral market tale about returning a los...,moral_tale,west_africa,doc_mor_001,The girl who returned the lost cowrie,0.338683
1,1002,Create a hero story of an orphan who saves fis...,hero_journey,east_africa,doc_her_003,Sail of the lake orphan,0.280876
2,1003,Create an East African tale of hare claiming t...,trickster,east_africa,doc_tri_003,Hare and the drum of thunder,0.274083
3,1004,Hyena jumps at the moon in water.,animal_fable,east_africa,doc_ani_001,Hyena and the moon's reflection,0.342270
4,1005,Tell a trickster tale of monkey guarding a hon...,trickster,central_africa,doc_tri_004,Monkey and the honey gate,0.267255
5,1006,Write an origin myth about stars spilling from...,origin_myth,southern_africa,doc_ori_004,Stars scattered from a calabash,0.294045
6,1007,Grandmother sings and the river bends — myth.,origin_myth,central_africa,doc_ori_002,How the river learned to bend,0.342014
7,1008,"Brothers, one path, drought — what lesson?",moral_tale,southern_africa,doc_mor_002,Two brothers and one path,0.363110
8,1009,Tell a hero tale of a drummer who brings rain ...,hero_journey,west_africa,doc_her_002,The drummer who carried rain,0.307375
9,1010,Harvest dance drum picks the leader.,community_wisdom,east_africa,doc_com_004,The drum that chose the dancer,0.339578


In [13]:
for i, row in test.iterrows():

    scores = similarities[i]

    top_indices = np.argsort(scores)[::-1][:3]

    print("=" * 80)
    print("Prompt:", row["prompt"])
    print("Theme:", row["theme"])
    print("Region:", row["culture_region"])

    for rank, idx in enumerate(top_indices, 1):

        doc = documents.iloc[idx]

        print(
            f"{rank}. {doc['title']} "
            f"(score={scores[idx]:.4f})"
        )

Prompt: Tell a moral market tale about returning a lost cowrie shell.
Theme: moral_tale
Region: west_africa
1. The girl who returned the lost cowrie (score=0.3387)
2. The pot that would not boil for greed (score=0.0742)
3. The lie that grew thorns (score=0.0732)
Prompt: Create a hero story of an orphan who saves fishermen in a squall.
Theme: hero_journey
Region: east_africa
1. Sail of the lake orphan (score=0.2809)
2. The drummer who carried rain (score=0.0963)
3. The salt bearer (score=0.0830)
Prompt: Create an East African tale of hare claiming thunder for himself.
Theme: trickster
Region: east_africa
1. Hare and the drum of thunder (score=0.2741)
2. Sail of the lake orphan (score=0.0988)
3. The pot that would not boil for greed (score=0.0897)
Prompt: Hyena jumps at the moon in water.
Theme: animal_fable
Region: east_africa
1. Hyena and the moon's reflection (score=0.3423)
2. Weaverbird's borrowed feathers (score=0.1082)
3. Elephant and the firefly council (score=0.0776)
Prompt: Tell

In [14]:
submission_rows = []

for i, row in test.iterrows():

    scores = similarities[i]

    best_index = np.argmax(scores)

    best_document = documents.iloc[best_index]

    submission_rows.append({
        "PromptId": row["PromptId"],
        "Story": best_document["text"]
    })

submission = pd.DataFrame(submission_rows)

In [15]:
display(submission)

,PromptId,Story
0,1001,A merchant dropped a cowrie shell in the marke...
1,1002,An orphan mended fishermen's nets without pay ...
2,1003,Hare found a hollow log that echoed like thund...
3,1004,Hyena saw the moon in a still pond and leapt t...
4,1005,Monkey promised to guard the hive while bees m...
5,1006,A daughter carried night in a covered calabash...
6,1007,Long ago the river ran straight and forgot the...
7,1008,Two brothers inherited one path to the grazing...
8,1009,"When clouds forgot the village, a young drumme..."
9,1010,Villagers argued whose son would lead the harv...


In [16]:
submission.to_csv(
    "/kaggle/working/submission_retrieval.csv",
    index=False
)